# Boids Flocking Influence Field

A JAX flock simulation where social forces are driven by sampled influence fields.

In [1]:
import asyncio

import jax
import jax.numpy as jnp
import ipywidgets as widgets
from IPython.display import display
from ipycanvas import Canvas, hold_canvas
from jax.scipy.ndimage import map_coordinates

from flock_sim import (
    alignment_field,
    boundary_field,
    cohesion_field,
    initialize_flock,
    make_update_step,
    separation_field,
    simulation_to_canvas,
)

In [6]:
population = 200
simulation_grid_size = 128
render_grid_size = 600

dt = 0.016
speed = 1.0

separation_strength = 2.0
cohesion_strength = 0.25
alignment_strength = 0.5
boundary_margin = 0.2
boundary_strength = 8.0
turn_rate = 0.2
sigma = 0.08
cohesion_sigma = 0.2
alignment_sigma = 0.16

rng_key = jax.random.key(10)
flock = initialize_flock(rng_key, population)

update_step = make_update_step(
    simulation_grid_size=simulation_grid_size,
    dt=dt,
    speed=speed,
    separation_strength=separation_strength,
    turn_rate=turn_rate,
    sigma=sigma,
    cohesion_strength=cohesion_strength,
    cohesion_sigma=cohesion_sigma,
    alignment_strength=alignment_strength,
    alignment_sigma=alignment_sigma,
    boundary_margin=boundary_margin,
    boundary_strength=boundary_strength,
)

In [4]:
bird_canvas = Canvas(width=render_grid_size, height=render_grid_size)
field_canvases = {}
field_tabs = widgets.Tab()


def field_layers(flock):
    separation = separation_field(
        flock=flock,
        simulation_grid_size=simulation_grid_size,
        sigma=sigma,
        strength=separation_strength,
    )
    cohesion = cohesion_field(
        flock=flock,
        simulation_grid_size=simulation_grid_size,
        sigma=cohesion_sigma,
        strength=cohesion_strength,
    )
    alignment = alignment_field(
        flock=flock,
        simulation_grid_size=simulation_grid_size,
        sigma=alignment_sigma,
        strength=alignment_strength,
    )
    boundary = boundary_field(
        simulation_grid_size=simulation_grid_size,
        margin=boundary_margin,
        strength=boundary_strength,
    )
    layers = {
        "Separation": separation,
        "Cohesion": cohesion,
        "Alignment": alignment,
        "Boundary": boundary,
    }
    return {"Combined": sum(layers.values()), **layers}


def ensure_field_tabs(layers):
    for name in layers:
        if name not in field_canvases:
            field_canvases[name] = Canvas(width=render_grid_size, height=render_grid_size)
    names = tuple(layers.keys())
    field_tabs.children = tuple(field_canvases[name] for name in names)
    for index, name in enumerate(names):
        field_tabs.set_title(index, name)


def hsv_to_rgb(hue, saturation, value):
    offsets = jnp.array([0.0, 4.0, 2.0])
    channels = jnp.abs((hue[..., None] * 6.0 + offsets) % 6.0 - 3.0) - 1.0
    rgb = jnp.clip(channels, 0.0, 1.0)
    return value[..., None] * (1.0 - saturation + saturation * rgb)


def draw_vector_field(canvas, field):
    rows = jnp.linspace(0, simulation_grid_size - 1, render_grid_size)
    cols = jnp.linspace(0, simulation_grid_size - 1, render_grid_size)
    grid_rows, grid_cols = jnp.meshgrid(rows, cols, indexing="ij")
    coords = jnp.stack([grid_rows, grid_cols])
    x = map_coordinates(field[:, :, 0], coords, order=1)
    y = map_coordinates(field[:, :, 1], coords, order=1)
    magnitude = jnp.sqrt(x * x + y * y)
    magnitude = jnp.log1p(magnitude)
    magnitude = magnitude / jnp.maximum(jnp.max(magnitude), 1e-8)
    hue = (jnp.arctan2(y, x) + jnp.pi) / (2.0 * jnp.pi)
    value = 0.12 + 0.88 * magnitude
    rgb = hsv_to_rgb(hue, 0.85, value)
    alpha = jnp.full_like(magnitude, 255, dtype=jnp.uint8)
    image = jnp.concatenate([(rgb * 255).astype(jnp.uint8), alpha[..., None]], axis=-1)
    canvas.put_image_data(jax.device_get(image), 0, 0)
    draw_field_arrows(canvas, field)


def draw_field_arrows(canvas, field, arrow_count=32):
    points = jnp.linspace(0, simulation_grid_size - 1, arrow_count)
    points = jnp.round(points).astype(jnp.int32)
    grid_rows, grid_cols = jnp.meshgrid(points, points, indexing="ij")
    vectors = field[grid_rows, grid_cols]
    lengths = jnp.linalg.norm(vectors, axis=-1)
    max_length = jnp.maximum(jnp.max(lengths), 1e-8)
    directions = vectors / jnp.maximum(lengths[..., None], 1e-8)
    starts_x = grid_cols * (render_grid_size - 1) / (simulation_grid_size - 1)
    starts_y = grid_rows * (render_grid_size - 1) / (simulation_grid_size - 1)
    arrow_lengths = 5.0 + 18.0 * jnp.sqrt(lengths / max_length)
    ends_x = starts_x + directions[:, :, 0] * arrow_lengths
    ends_y = starts_y + directions[:, :, 1] * arrow_lengths
    canvas.stroke_style = "rgba(255, 255, 255, 0.55)"
    canvas.line_width = 1.25
    for start_x, start_y, end_x, end_y, length in zip(
        jax.device_get(starts_x).ravel(),
        jax.device_get(starts_y).ravel(),
        jax.device_get(ends_x).ravel(),
        jax.device_get(ends_y).ravel(),
        jax.device_get(lengths).ravel(),
    ):
        if length > 1e-8:
            canvas.stroke_line(float(start_x), float(start_y), float(end_x), float(end_y))


def draw_birds(canvas, flock):
    points = simulation_to_canvas(flock.positions, render_grid_size)
    points = jax.device_get(points)
    canvas.clear()
    canvas.fill_style = "rgb(8, 10, 14)"
    canvas.fill_rect(0, 0, render_grid_size, render_grid_size)
    canvas.fill_style = "rgba(0, 0, 0, 0.8)"
    for x, y in points:
        canvas.fill_circle(float(x), float(y), 4.0)
    canvas.fill_style = "white"
    for x, y in points:
        canvas.fill_circle(float(x), float(y), 2.4)


def draw_frame(flock):
    layers = field_layers(flock)
    ensure_field_tabs(layers)
    for name, field in layers.items():
        canvas = field_canvases[name]
        with hold_canvas(canvas):
            draw_vector_field(canvas, field)
    with hold_canvas(bird_canvas):
        draw_birds(bird_canvas, flock)

In [5]:
simulation_running = False
simulation_paused = False
simulation_task = None

toggle_button = widgets.Button(description="Start", button_style="success")
display(toggle_button)


def set_button_state(running, paused=False):
    if not running:
        toggle_button.description = "Start"
        toggle_button.button_style = "success"
    elif paused:
        toggle_button.description = "Resume"
        toggle_button.button_style = "info"
    else:
        toggle_button.description = "Pause"
        toggle_button.button_style = "warning"


async def run_simulation():
    global flock, simulation_running, simulation_paused
    simulation_running = True
    set_button_state(running=True, paused=simulation_paused)

    try:
        while simulation_running:
            if not simulation_paused:
                flock = update_step(flock)
                draw_frame(flock)
            await asyncio.sleep(dt)
    except asyncio.CancelledError:
        pass
    finally:
        simulation_running = False
        simulation_paused = False
        set_button_state(running=False)


def start_simulation():
    global simulation_running, simulation_paused, simulation_task
    if simulation_task is not None and not simulation_task.done():
        return
    simulation_running = True
    simulation_paused = False
    simulation_task = asyncio.create_task(run_simulation())
    set_button_state(running=True)


def pause_simulation():
    global simulation_paused
    if not simulation_running:
        return
    simulation_paused = not simulation_paused
    set_button_state(running=True, paused=simulation_paused)


def toggle_simulation(_=None):
    if simulation_running:
        pause_simulation()
    else:
        start_simulation()


def stop_simulation(_=None):
    global simulation_running, simulation_paused, simulation_task
    simulation_running = False
    simulation_paused = False
    if simulation_task is not None and not simulation_task.done():
        simulation_task.cancel()
    set_button_state(running=False)


toggle_button.on_click(toggle_simulation)
draw_frame(flock)
display(widgets.HBox([bird_canvas, field_tabs]))

Button(button_style='success', description='Start', style=ButtonStyle())